In [1]:
!pip install faker xgboost lightgbm shap scikit-learn pandas numpy matplotlib seaborn -q

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, recall_score
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
N_SAMPLES = 20000
FRAUD_RATIO = 0.18
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
fake = Faker('en_IN')
Faker.seed(RANDOM_STATE)

def generate_harder_data(n_samples=20000, fraud_ratio=0.18):
    data = []
    shared_devices = [fake.uuid4() for _ in range(150)]
    shared_ips = [fake.ipv4() for _ in range(110)]
    shared_phones = [fake.phone_number() for _ in range(130)]

    for i in range(n_samples):
        is_fraud = 1 if np.random.rand() < fraud_ratio else 0
        age = np.random.randint(20, 55)
        education = np.random.choice(['High School', 'Graduate', 'Post Graduate'], p=[0.3, 0.5, 0.2])

        if is_fraud:
            difficulty = np.random.choice(['easy', 'medium', 'hard'], p=[0.30, 0.25, 0.45])

            if difficulty == 'easy':
                work_exp = max(0, age - np.random.randint(7, 14))
                job = np.random.choice(['Student', 'Unemployed'])
                income = np.random.randint(12, 35) * 100000
                device = np.random.choice(shared_devices)
                ip = np.random.choice(shared_ips)
                phone = np.random.choice(shared_phones)
                address_changes = np.random.randint(3, 8)
                num_apps = np.random.randint(4, 8)
            elif difficulty == 'medium':
                work_exp = max(0, age - np.random.randint(15, 21))
                job = np.random.choice(['Salaried', 'Self-employed', 'Student'])
                income = np.random.randint(8, 25) * 100000
                device = np.random.choice(shared_devices) if np.random.rand() < 0.55 else fake.uuid4()
                ip = np.random.choice(shared_ips) if np.random.rand() < 0.4 else fake.ipv4()
                phone = fake.phone_number()
                address_changes = np.random.randint(2, 5)
                num_apps = np.random.randint(2, 5)
            else:  # hard
                work_exp = max(0, age - np.random.randint(18, 24))
                job = np.random.choice(['Salaried', 'Self-employed'])
                income = np.random.randint(5, 18) * 100000
                device = np.random.choice(shared_devices) if np.random.rand() < 0.25 else fake.uuid4()
                ip = fake.ipv4()
                phone = fake.phone_number()
                address_changes = np.random.randint(0, 3)
                num_apps = np.random.randint(1, 3)
            fraud_difficulty = difficulty
        else:
            work_exp = max(0, age - np.random.randint(19, 27))
            if np.random.rand() < 0.08:
                work_exp = max(0, work_exp - np.random.randint(1, 3))
            job = np.random.choice(['Salaried', 'Self-employed', 'Government', 'Student'], p=[0.55, 0.25, 0.1, 0.1])
            income = np.random.randint(2, 22) * 100000
            if job == 'Student':
                income = np.random.randint(0, 5) * 100000
            device = fake.uuid4()
            ip = fake.ipv4()
            phone = fake.phone_number()
            address_changes = np.random.randint(0, 3)
            num_apps = np.random.randint(1, 3)
            fraud_difficulty = 'Genuine'

        data.append({
            'applicant_id': f'APP{100000 + i}',
            'age': age,
            'work_experience': work_exp,
            'declared_income': income,
            'job_category': job,
            'education': education,
            'device_id': device,
            'ip_address': ip,
            'phone_number': phone,
            'address_change_count': address_changes,
            'num_applications': num_apps,
            'fraud_difficulty': fraud_difficulty,
            'label': is_fraud
        })

    return pd.DataFrame(data)

df = generate_harder_data(N_SAMPLES, FRAUD_RATIO)
print("Data shape:", df.shape)
print(df['label'].value_counts(normalize=True))
print("\nFraud Difficulty:")
print(df['fraud_difficulty'].value_counts())
df.head()

Data shape: (20000, 13)
label
0    0.82055
1    0.17945
Name: proportion, dtype: float64

Fraud Difficulty:
fraud_difficulty
Genuine    16411
hard        1650
easy        1042
medium       897
Name: count, dtype: int64


,applicant_id,age,work_experience,declared_income,job_category,education,device_id,ip_address,phone_number,address_change_count,num_applications,fraud_difficulty,label
0,APP100000,48,22,2000000,Salaried,High School,7f53a88e-4efc-4248-a10b-1a477faeb748,89.42.225.70,7339500479,2,1,Genuine,0
1,APP100001,43,20,700000,Government,Graduate,ac2a4f71-a77d-45f1-97c4-fe9ce1e0fffc,16.66.253.65,+916245650609,1,2,Genuine,0
2,APP100002,52,27,1100000,Salaried,Graduate,8da00495-5cce-4ba9-b1d4-ee098fd33afc,56.185.240.105,7849912165,2,2,Genuine,0
3,APP100003,22,4,600000,Salaried,Post Graduate,8ece1128-56ce-4b42-bc9e-1dcbcb7e6268,205.39.117.165,+919868000257,1,2,hard,1
4,APP100004,45,23,1300000,Self-employed,High School,9b33e3a4-2620-49a9-b4fd-33d184f2fd0f,212.121.21.251,09526949588,0,2,Genuine,0


In [4]:
# ===============================================================
#  Rules + Features + Identity Coherence Score
# ===============================================================

def apply_consistency_rules(df):
    df = df.copy()

    df['rule_age_exp'] = ((df['work_experience'] > (df['age'] - 17)) |
                          ((df['age'] <= 23) & (df['work_experience'] >= 5))).astype(int)

    df['rule_income_job'] = (((df['job_category'].isin(['Student', 'Unemployed'])) &
                              (df['declared_income'] > 750000))).astype(int)

    df['rule_velocity'] = (df['num_applications'] >= 4).astype(int)
    df['rule_address'] = (df['address_change_count'] >= 4).astype(int)

    df['rule_score'] = df[['rule_age_exp', 'rule_income_job', 'rule_velocity', 'rule_address']].sum(axis=1)
    return df

df = apply_consistency_rules(df)
print("Rules applied successfully!")
print(df[['rule_age_exp', 'rule_income_job', 'rule_velocity', 'rule_address', 'rule_score']].head())


# ---------------------------------------------------------------
# Feature Engineering (Basic version - leakage free later in CV)
# ---------------------------------------------------------------
def create_basic_features(df):
    df = df.copy()

    df['device_share'] = df.groupby('device_id')['applicant_id'].transform('count')
    df['ip_share'] = df.groupby('ip_address')['applicant_id'].transform('count')
    df['phone_share'] = df.groupby('phone_number')['applicant_id'].transform('count')

    df['device_share_log'] = np.log1p(df['device_share'])
    df['ip_share_log'] = np.log1p(df['ip_share'])
    df['phone_share_log'] = np.log1p(df['phone_share'])

    df['age_exp_gap'] = df['age'] - df['work_experience']
    df['income_per_exp'] = df['declared_income'] / (df['work_experience'] + 1)
    df['income_per_age'] = df['declared_income'] / df['age']

    return df

df = create_basic_features(df)
print("\nBasic features created!")


# ---------------------------------------------------------------
# Identity Coherence Score (ICS)
# ---------------------------------------------------------------
def calculate_identity_coherence_score(df):
    df = df.copy()

    # 1. Demographic Coherence
    demo_penalty = (
        df['rule_age_exp'] * 0.40 +
        df['rule_income_job'] * 0.35 +
        (df['age_exp_gap'] < 10).astype(int) * 0.15 +
        (df['income_per_exp'] > df['income_per_exp'].quantile(0.95)).astype(int) * 0.10
    )
    demo_penalty = np.clip(demo_penalty, 0, 1)
    df['demographic_coherence'] = 1 - demo_penalty

    # 2. Relational Coherence
    rel_penalty = (
        0.50 * df['device_share_log'] +
        0.30 * df['ip_share_log'] +
        0.20 * df['phone_share_log']
    )
    scaler = MinMaxScaler()
    rel_penalty_norm = scaler.fit_transform(rel_penalty.values.reshape(-1, 1)).flatten()
    df['relational_coherence'] = 1 - rel_penalty_norm

    # 3. Behavioral Coherence
    behav_penalty = (
        0.60 * (df['num_applications'] >= 4).astype(int) +
        0.40 * (df['address_change_count'] >= 3).astype(int)
    )
    behav_penalty = np.clip(behav_penalty, 0, 1)
    df['behavioral_coherence'] = 1 - behav_penalty

    # Final ICS
    w1, w2, w3 = 0.40, 0.35, 0.25
    df['identity_coherence_score'] = (
        w1 * df['demographic_coherence'] +
        w2 * df['relational_coherence'] +
        w3 * df['behavioral_coherence']
    )

    # Risk version
    df['identity_risk_from_ics'] = 1 - df['identity_coherence_score']

    return df

df = calculate_identity_coherence_score(df)

print("\nIdentity Coherence Score calculated!")
print(df[['applicant_id', 'demographic_coherence', 'relational_coherence',
          'behavioral_coherence', 'identity_coherence_score', 'identity_risk_from_ics']].head(10))

print("\nICS Statistics:")
print(df['identity_coherence_score'].describe())

Rules applied successfully!
   rule_age_exp  rule_income_job  rule_velocity  rule_address  rule_score
0             0                0              0             0           0
1             0                0              0             0           0
2             0                0              0             0           0
3             0                0              0             0           0
4             0                0              0             0           0

Basic features created!

Identity Coherence Score calculated!
  applicant_id  demographic_coherence  relational_coherence  \
0    APP100000                   1.00              1.000000   
1    APP100001                   1.00              1.000000   
2    APP100002                   1.00              1.000000   
3    APP100003                   1.00              1.000000   
4    APP100004                   1.00              1.000000   
5    APP100005                   0.25              0.079116   
6    APP100006          

In [5]:
# ===============================================================
#  Model Training + Evaluation
# ===============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, recall_score

# Encode categorical columns
le_job = LabelEncoder()
le_edu = LabelEncoder()
df['job_enc'] = le_job.fit_transform(df['job_category'])
df['edu_enc'] = le_edu.fit_transform(df['education'])

FEATURE_COLS = [
    'age', 'work_experience', 'declared_income', 'job_enc', 'edu_enc',
    'address_change_count', 'num_applications',
    'device_share_log', 'ip_share_log', 'phone_share_log',
    'age_exp_gap', 'income_per_exp', 'income_per_age', 'rule_score',
    'identity_coherence_score'          # <-- ICS added as feature
]

X = df[FEATURE_COLS]
y = df['label']

print("Features used:", FEATURE_COLS)
print("X shape:", X.shape)

# ---------------------------------------------------------------
# Stratified 5-Fold Cross Validation
# ---------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(df))
oof_lgb = np.zeros(len(df))

fold = 1
for train_idx, val_idx in skf.split(X, y):
    print(f"Training Fold {fold}...")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        eval_metric='auc',
        n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        n_jobs=-1
    )
    lgb_model.fit(X_train, y_train)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]

    fold += 1

df['xgb_proba'] = oof_xgb
df['lgb_proba'] = oof_lgb

# ---------------------------------------------------------------
# Results
# ---------------------------------------------------------------
print("\n========== OVERALL RESULTS ==========")
print("XGBoost ROC-AUC :", round(roc_auc_score(y, oof_xgb), 4))
print("XGBoost PR-AUC  :", round(average_precision_score(y, oof_xgb), 4))
print("LightGBM ROC-AUC:", round(roc_auc_score(y, oof_lgb), 4))
print("LightGBM PR-AUC :", round(average_precision_score(y, oof_lgb), 4))

print("\n========== Classification Report (XGBoost) ==========")
print(classification_report(y, (oof_xgb > 0.5).astype(int)))

# ---------------------------------------------------------------
# Performance on Easy vs Hard Fraud
# ---------------------------------------------------------------
print("\n========== RECALL BY FRAUD DIFFICULTY ==========")
for diff in ['easy', 'medium', 'hard']:
    mask = (df['fraud_difficulty'] == diff)
    if mask.sum() > 0:
        rec = recall_score(df.loc[mask, 'label'], (df.loc[mask, 'xgb_proba'] > 0.5).astype(int))
        print(f"{diff.upper():<8} Fraud Recall: {rec:.4f}   (Samples: {mask.sum()})")

print("\nStep 5 Completed Successfully!")

Features used: ['age', 'work_experience', 'declared_income', 'job_enc', 'edu_enc', 'address_change_count', 'num_applications', 'device_share_log', 'ip_share_log', 'phone_share_log', 'age_exp_gap', 'income_per_exp', 'income_per_age', 'rule_score', 'identity_coherence_score']
X shape: (20000, 15)
Training Fold 1...
[LightGBM] [Info] Number of positive: 2872, number of negative: 13128
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001253 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 933
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.179500 -> initscore=-1.519739
[LightGBM] [Info] Start training from score -1.519739
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

In [6]:
# ===============================================================
#  Final Risk Score + Save Everything
# ===============================================================

# Final Identity Risk Score (using XGBoost probability + ICS)
df['final_identity_risk_score'] = (
    0.70 * df['xgb_proba'] +
    0.30 * df['identity_risk_from_ics']
)

# Risk Category
df['risk_category'] = pd.cut(
    df['final_identity_risk_score'],
    bins=[-0.01, 0.30, 0.60, 1.01],
    labels=['Low', 'Medium', 'High']
)

print("Risk Category Distribution:")
print(df['risk_category'].value_counts())
print("\n")

print("Sample High Risk Cases:")
print(df[df['risk_category'] == 'High'][['applicant_id', 'age', 'work_experience',
                                          'declared_income', 'job_category',
                                          'final_identity_risk_score', 'fraud_difficulty']].head(8))

# Save final dataset
df.to_csv('application_stage_final_20k.csv', index=False)
print("\nFile saved: application_stage_final_20k.csv")

# Save model (optional)
xgb_model.fit(X, y)   # Fit on full data
xgb_model.save_model('xgb_identity_model.json')
print("Model saved: xgb_identity_model.json")

print("\n========== APPLICATION STAGE COMPLETED SUCCESSFULLY ==========")

Risk Category Distribution:
risk_category
Low       14680
Medium     2823
High       2497
Name: count, dtype: int64


Sample High Risk Cases:
   applicant_id  age  work_experience  declared_income   job_category  \
3     APP100003   22                4           600000       Salaried   
5     APP100005   33               23          1700000     Unemployed   
37    APP100037   44               21          1400000  Self-employed   
40    APP100040   38               27          3200000     Unemployed   
46    APP100046   51               32          1600000       Salaried   
55    APP100055   54               38          1900000        Student   
59    APP100059   22                2          1500000  Self-employed   
65    APP100065   28                7          1700000  Self-employed   

    final_identity_risk_score fraud_difficulty  
3                    0.697143             hard  
5                    0.961657             easy  
37                   0.738962             hard  
40  